### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for
ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple
schema types and methods for enforcing structured output.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027F1B52CAD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027F1B52D550>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
from pydantic import BaseModel,Field

class movie(BaseModel):
    title:str=Field(description="The Title of the movie"),
    year:int=Field(description="the year movie released"),
    director:str=Field(description="the director of movie"),
    rating:float=Field(description="rating of movie")

In [3]:
model_with_structure=model.with_structured_output(movie)
model_with_structure

c:\Users\Tarunvarma\OneDrive\Desktop\langchain\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The Title of the movie'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\Tarunvarma\OneDrive\Desktop\langchain\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='the year movie released'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\Tarunvarma\OneDrive\Desktop\langchain\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='the director of movie'),) is not JSON serializable; excludi

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027F1B52CAD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027F1B52D550>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'movie', 'description': '', 'parameters': {'properties': {'title': {'type': 'string'}, 'year': {'type': 'integer'}, 'director': {'type': 'string'}, 'rating': {'description': 'rating of movie', 'type': 'number'}}, 'required': ['rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'},

In [4]:
response=model_with_structure.invoke("Give me details about the movie fight club")
response

movie(title='Fight Club', year=1999, director='David Fincher', rating=8.8)

### Message output alongside parsed structure 

In [5]:
from pydantic import BaseModel,Field

class movie(BaseModel):
    """A movie with details"""
    title:str=Field(description="The Title of the movie"),
    year:int=Field(description="the year movie released"),
    director:str=Field(description="the director of movie"),
    rating:float=Field(description="rating of movie")

model_with_structure=model.with_structured_output(movie,include_raw=True)

response=model_with_structure.invoke("provide details about inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There\'s a function called movie that requires a rating. The parameters include director, rating, title, and year. The user didn\'t specify the rating, but the function requires it. I need to figure out the rating for Inception. I remember that Inception has a high rating on IMDb, maybe around 8.8. The director is Christopher Nolan, and the release year was 2010. Let me confirm those details. Yeah, that\'s correct. So I\'ll use the movie function with the title "Inception", director "Christopher Nolan", year 2010, and rating 8.8. That should provide the user with the necessary information.\n', 'tool_calls': [{'id': 'q9ypm9sja', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_t

### Nested Structure

In [6]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
  name: str
  role: str

class MovieDetails(BaseModel):
  title: str
  year: int
  cast: list[Actor]
  genres: list[str]
  budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Fight Club")
response


MovieDetails(title='Fight Club', year=1999, cast=[Actor(name='Edward Norton', role='Tyler Durden'), Actor(name='Brad Pitt', role='The Narrator'), Actor(name='Meat Loaf', role='Bob')], genres=['Drama', 'Thriller'], budget=63.0)

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [7]:
from typing_extensions import TypedDict,Annotated

class movieDict(TypedDict):
    """ A movie details"""
    title:Annotated[str,...,"title of the movie"]
    year:Annotated[int,...,"the year movie released"]
    director:Annotated[str,...,"director of movie"]
    rating:Annotated[float,...,"movie rating out of 10"]

model_with_TypeDict=model.with_structured_output(movieDict)
response=model_with_TypeDict.invoke("provide details of avenger")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [8]:
from typing_extensions import TypedDict,Annotated

class Actor(TypedDict):
  name: str
  role: str

class MovieDetails(TypedDict):
  title: str
  year: int
  cast: list[Actor]
  genres: list[str]
  budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie endgame")
response


{'budget': 356000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Paul Rudd', 'role': 'Scott Lang / Ant-Man'},
  {'name': 'Brie Larson', 'role': 'Carol Danvers / Captain Marvel'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Endgame',
 'year': 2019}

In [9]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### Data Class

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")

In [25]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """ contact info of a person"""
    name: str = Field(description="name of the person")
    email: str = Field(description="email address of person")
    phone: str = Field(description="phone number of the person")

agent = create_agent(
    model,
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='cfc17687-93d5-469b-98c4-397c0964d7a3'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to extract contact information from the given string: "John Doe, john@example.com, (555) 123-4567". Let me break this down.\n\nFirst, I need to identify the different parts. The name is "John Doe", which is straightforward. The email address is "john@example.com". The phone number is "(555) 123-4567". \n\nNow, looking at the provided function, ContactInfo, the required parameters are name, email, and phone. All three are present here. The phone number is formatted with parentheses and a space, but the function doesn\'t specify any particular format for the phone number, just that it\'s a string. So I can include it as-is.\n\nI should make sure there are no typos. Let me double-check each part. Name: "John 

In [27]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """ contact info of a person"""
    name: str 
    email: str 
    phone: str 

agent=create_agent(
    model,
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')